# Computer-Use Model Server (Kaggle free GPU) - UI-TARS-1.5-7B

Runs ByteDance's UI-TARS-1.5-7B on the free T4x2 GPU and exposes it as an OpenAI-compatible API. Better at clicking and GUI tasks than the base Qwen model.~30GB usable)

**Setup:**

1. Settings -> Accelerator -> GPU T4 x2 -> Save
2. Run cells 1-4 (model loads, server starts on port 8000, Gradio UI opens on port 7860)
3. Run cell 5 in a separate terminal to open the tunnel
4. Paste the public URL into assistant_ui.html (Settings) on your PC

**Edit the system prompt:** open the Gradio UI (the tunnel URL with port 7860, or the Kaggle output link) and type in the System Prompt box. It saves live - no restart needed. Or POST to /prompt


In [ ]:
# Cell 1: Install dependencies (first run only, ~2 min)
!pip install -q transformers accelerate torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q fastapi uvicorn pydantic bitsandbytes gradio

In [ ]:
# Cell 2: Load UI-TARS-1.5-7B on the GPU (4-bit, fits T4x2 ~30GB usable)
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

MODEL_ID = "ByteDance-Seed/UI-TARS-1.5-7B"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    load_in_4bit=True,
    trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
print("Model loaded on", next(model.parameters()).device)

In [ ]:
# Cell 3: FastAPI server - OpenAI-compatible /v1/chat/completions + editable system prompt
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import uvicorn

app = FastAPI()

SYSTEM_PROMPT = (
    "You are a computer-use assistant. You see the user's screen and control their PC.\n\n"
    "Rules:\n"
    "- Prefer keyboard over mouse clicks whenever possible (typing, shortcuts, URLs).\n"
    "- Use clicks only when there is no keyboard way to do the task.\n"
    "- Confirm before any destructive action (delete, format, uninstall, shutdown).\n"
    "- Reply with one clear action at a time: what to type or where to click.\n"
    "- If a click misses, say so and suggest the keyboard alternative.\n"
    "- Keep answers short. No explanations unless asked.\n"
)

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = MODEL_ID
    messages: List[Message]
    max_tokens: int = 512
    temperature: float = 0.2

class PromptUpdate(BaseModel):
    prompt: str

@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    msgs = [Message(role="system", content=SYSTEM_PROMPT)] + req.messages
    prompt = "\n".join(f"{m.role}: {m.content}" for m in msgs)
    inputs = processor(text=prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=req.max_tokens, temperature=req.temperature)
    text = processor.decode(out[0], skip_special_tokens=True)
    reply = text[len(prompt):].strip() if text.startswith(prompt) else text
    return {
        "id": "kaggle-1",
        "object": "chat.completion",
        "choices": [{"index": 0, "message": {"role": "assistant", "content": reply}, "finish_reason": "stop"}]
    }

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}

@app.get("/prompt")
def get_prompt():
    return {"prompt": SYSTEM_PROMPT}

@app.post("/prompt")
def set_prompt(req: PromptUpdate):
    global SYSTEM_PROMPT
    SYSTEM_PROMPT = req.prompt
    return {"status": "ok", "prompt": SYSTEM_PROMPT}

print("Server defined. Run the next cell to start it.")

In [ ]:
# Cell 4: Start the server (runs until stopped) + Gradio UI for editing the system prompt
import gradio as gr
import threading

def save_prompt(text):
    global SYSTEM_PROMPT
    SYSTEM_PROMPT = text
    return "Saved. Every new message now uses this prompt."

ui = gr.Interface(
    fn=save_prompt,
    inputs=gr.Textbox(value=SYSTEM_PROMPT, lines=12, label="System Prompt - edit and hit Submit"),
    outputs=gr.Textbox(label="Status"),
    title="Computer-Use Server - System Prompt Editor",
    description="Change the defaults the model sees on every message. No restart needed."
)
threading.Thread(target=lambda: ui.launch(server_name="0.0.0.0", server_port=7860, share=False), daemon=True).start()
print("Gradio UI starting on port 7860")
uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
# Cell 5: Tunnel - run in a SEPARATE terminal so the server keeps running
# cloudflared needs no account. It prints a public https URL for port 8000 (the API).
# The Gradio UI is on port 7860 - open it from the Kaggle output link or add :7860 to the tunnel host if you expose it too.
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8000